In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ulmo
import warnings
warnings.filterwarnings("ignore")

# In this script, we will download daymet data and compare it with our input meteorological forcing data from ERA5.

In [ ]:
# you will need to change the minimum 
# and maximum values for the lat/lon
# based on your study domain

lon_list = np.arange(-73.7,-73,0.01)
lat_list = np.arange(41.3,42.7,0.01)

# annual_mean_precip_daymet = xr.DataArray(np.full([len(lat_list),len(lon_list)],np.NaN),
#                                          dims=['lat','lon'],
#                                          coords = {'lat':lat_list,
#                                                    'lon':lon_list})

### We use `ulmo.nasa.daymet.get_variables()` to see what variables we can download from daymet dataset

In [ ]:
ulmo.nasa.daymet.get_variables()

### Next, we create a folder to store the data for each grid cells

In [ ]:
import os

In [ ]:
daymet_grid_dir = "daymet_grid/"
if not os.path.isdir(daymet_grid_dir):
    os.mkdir(daymet_grid_dir)

### Here, we defined a function to 1) download the daymet data, and 2) calculate their mean monthly values, and 3) store them in the folder created above

In [ ]:
def calculate_daymet_precip_temp(input_list):
    [latitude,longitude,ilat,ilon, sty,edy] = input_list
    met_grid = ulmo.nasa.daymet.get_daymet_singlepixel(latitude, longitude, 
                                                      variables=['prcp','tmax','tmin'], 
                                                      years=[x for x in np.arange(sty,edy)], 
                                                      as_dataframe=True)
    month_mean_precip = met_grid['prcp'].resample('M').sum()
    my_month_mean_precip = month_mean_precip.groupby(month_mean_precip.index.month).mean()
    month_mean_dmax_temp = met_grid['tmax'].groupby(met_grid.index.month).mean()
    month_mean_dmin_temp = met_grid['tmin'].groupby(met_grid.index.month).mean()
    month_mean_dmean_temp = (month_mean_dmax_temp+month_mean_dmin_temp)/2
    df = pd.DataFrame(np.transpose([my_month_mean_precip,month_mean_dmean_temp]),
                  columns = ['prcp_mm_month','tair_deg_C'],
                  index = np.arange(1,13))
    df.to_csv(daymet_grid_dir+"daymet.%s_%s.csv"%("%0.2f"%(latitude),
                                                  "%0.2f"%(longitude)))

# Here we introduce parallelization called `multiprocessing`

This package allows you to parallelize the redundant job and greatly improve the efficiency of your scripts!

In [ ]:
from multiprocessing import Pool

# 10 here means we use 10 workers
pool = Pool(processes=10)

In [ ]:
job_list = []
start_year = 1990
end_year = 2005
for ilat,latitude in enumerate(lat_list):
    for ilon, longitude in enumerate(lon_list):
        job_list.append([np.around(latitude,2),np.around(longitude,2),
                         ilat,ilon,start_year,end_year])

In [ ]:
pool.map(calculate_daymet_precip_temp, job_list)

In [ ]:
# for ilat,latitude in enumerate(lat_list):
#     for ilon, longitude in enumerate(lon_list):
#         if np.isnan(annual_mean_precip_daymet[ilat,ilon]):
#             precip_grid = ulmo.nasa.daymet.get_daymet_singlepixel(latitude, longitude, variables=['prcp'], 
#                                                     years=[1990,2005], as_dataframe=True)
#             annual_mean_precip = precip_grid['prcp'].groupby(precip_grid.index.year).sum().mean()
#             annual_mean_precip_daymet[ilat,ilon] = annual_mean_precip

In [ ]:
# met_grid = ulmo.nasa.daymet.get_daymet_singlepixel(latitude, longitude, 
#                                                       variables=['prcp','tmax','tmin'], 
#                                                       years=[x for x in np.arange(1990,2006)], 
#                                                       as_dataframe=True)
# month_mean_precip = met_grid['prcp'].resample('M').sum()
# my_month_mean_precip = month_mean_precip.groupby(month_mean_precip.index.month).mean()
# month_mean_dmax_temp = met_grid['tmax'].groupby(met_grid.index.month).mean()
# month_mean_dmin_temp = met_grid['tmin'].groupby(met_grid.index.month).mean()
# month_mean_dmean_temp = (month_mean_dmax_temp+month_mean_dmin_temp)/2

In [ ]:
# df = pd.DataFrame(np.transpose([my_month_mean_precip,month_mean_dmean_temp]),
#                   columns = ['prcp_mm_month','tair_deg_C'],
#                   index = np.arange(1,13))

In [ ]:
# df